## Fase 3: Función de Predicción y Preparación para Despliegue

In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
import joblib

# Cargar el modelo 
model = tf.keras.models.load_model("modelo_lstm.h5")

# Cargar dataset preparado
df = pd.read_csv("dataset_preparado.csv", parse_dates=["created_at"])
print(df.shape)
df.head()


(4520, 19)


,product_id,created_at,ventas_diarias,quantity_on_hand,reorder_point,optimal_stock_level,average_daily_usage,stock_status,anio,mes,dia_semana,fin_de_semana,venta_t-1,venta_t-2,venta_t-3,venta_t-4,venta_t-5,venta_t-6,venta_t-7
0,PROD-001,2024-03-08,0,124,34,46,1.86,1,2024,3,4,False,0.0,0.0,4.0,3.0,6.0,0.0,0.0
1,PROD-001,2024-03-09,0,124,64,82,1.86,1,2024,3,5,True,0.0,0.0,0.0,4.0,3.0,6.0,0.0
2,PROD-001,2024-03-10,0,124,28,42,1.00,1,2024,3,6,True,0.0,0.0,0.0,0.0,4.0,3.0,6.0
3,PROD-001,2024-03-11,5,119,44,69,1.29,1,2024,3,0,False,0.0,0.0,0.0,0.0,0.0,4.0,3.0
4,PROD-001,2024-03-12,0,119,42,75,0.71,1,2024,3,1,False,5.0,0.0,0.0,0.0,0.0,0.0,4.0


In [4]:
# Variables usadas en el modelo 
FEATURES = [
    "ventas_diarias", "quantity_on_hand", "reorder_point", "optimal_stock_level",
    "average_daily_usage", "stock_status", "dia_semana", "fin_de_semana"
]
N_STEPS = 7

scaler = StandardScaler()
scaler.fit(df[FEATURES])

def build_sequence_for_product(product_id, target_date, df, n_steps=N_STEPS):
    """
    Construye la secuencia de los últimos n_steps días previos a target_date
    para el producto dado.
    """
    df_p = df[df["product_id"] == product_id].sort_values("created_at").copy()
    
    # Convertir a datetime si no lo está
    df_p["created_at"] = pd.to_datetime(df_p["created_at"])
    
    # Obtener los n_steps días previos a la fecha indicada
    mask = df_p["created_at"] < target_date
    df_prev = df_p[mask].tail(n_steps)
    
    if len(df_prev) < n_steps:
        raise ValueError(f"No hay suficientes datos previos para {product_id}. Se requieren {n_steps} días.")
    
    # Escalar las features
    seq_scaled = scaler.transform(df_prev[FEATURES])
    
    return np.expand_dims(seq_scaled, axis=0)


In [5]:
def predict_demand(product_id: str, date: str):
    """
    Predice las ventas diarias esperadas para un producto en una fecha dada.

    Parámetros:
        product_id (str): ID del producto (ej. 'PROD-005')
        date (str o datetime): Fecha objetivo (formato 'YYYY-MM-DD')

    Retorna:
        dict: { "product_id": ..., "fecha_objetivo": ..., "prediccion_ventas": ... }
    """
    if isinstance(date, str):
        date = datetime.strptime(date, "%Y-%m-%d")
    
    # Generar secuencia
    X_input = build_sequence_for_product(product_id, date, df)
    
    # Predecir
    pred = model.predict(X_input)
    pred_val = float(pred[0][0])  # conversión a float simple
    
    return {
        "product_id": product_id,
        "fecha_objetivo": date.strftime("%Y-%m-%d"),
        "prediccion_ventas": round(pred_val, 2)
    }


In [10]:
ejemplo_prod = df["product_id"].unique()[1]
fecha_pred = df["created_at"].max() + timedelta(days=0)

resultado = predict_demand(ejemplo_prod, fecha_pred)
print(resultado)



1/1 [==============================] - 0s 40ms/step
{'product_id': 'PROD-002', 'fecha_objetivo': '2024-10-19', 'prediccion_ventas': 0.64}
